<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.2: 生成器: 集合
**上一节: [生成器: 参数](3.1_parameters.ipynb)**<br>
**下一节: [插曲: Chisel 标准库](3.2_interlude.ipynb)**


## 动机
生成器经常需要处理可变数量的对象，无论是 IOs、模块还是测试向量。
集合是处理这种情况的重要构建块。
本模块将介绍 Scala 集合以及如何将它们与 Chisel 生成器一起使用。

## 设置

In [1]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

path: String = "/Users/zhaochanglong/Documents/chisel-bootcamp-zh/source/load-ivy.sc"

注意我们在这里添加了一个新的导入，因为 `mutable.ArrayBuffer` 位于 `scala.collections` 中。

In [2]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test
import scala.collection._

import chisel3._

import chisel3.util._

import chisel3.tester._

import chisel3.tester.RawTester.test

import scala.collection._

---
# 生成器和集合<a name="generators-and-collections"></a> 
在本节中，我们将重点介绍*生成器*的概念以及使用 Scala 集合作为实现它们的工具。
我们不再将 Chisel 代码视为电路的*实例*，即特定电路的描述，
而是将其视为电路的生成器。

我们将从考虑之前练习中的 FIR 滤波器开始。

In [4]:
class My4ElementFir(b0: Int, b1: Int, b2: Int, b3: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val out = Output(UInt(8.W))
  })

  val x_n1 = RegNext(io.in, 0.U)
  val x_n2 = RegNext(x_n1, 0.U)
  val x_n3 = RegNext(x_n2, 0.U)
  io.out := io.in * b0.U(8.W) + x_n1 * b1.U(8.W) +
    x_n2 * b2.U(8.W) + x_n3 * b3.U(8.W)
}


defined class My4ElementFir

这个电路是生成器的一个简单案例，因为它可以生成具有不同系数的 4 抽头滤波器版本。
但是如果我们希望电路有更多抽头怎么办？我们将分几个步骤来完成这个任务。

- 构建一个抽头可配置 FIR 的软件*黄金模型*。
- 重新设计我们的测试以使用这个模型，并确认它有效。
- 重构我们的 My4ElementFir 以允许可配置的抽头数量。
- 使用我们新的测试框架测试新电路。

<span style="color:blue">**示例: FIR 黄金模型**</span><br><a name="fir-golden-model"></a> 
下面是一个 FIR 电路的 Scala 软件实现。

In [ ]:
/**
  * 一个具有任意抽头数量的 FIR 滤波器的简单实现。
  */
class ScalaFirFilter(taps: Seq[Int]) {
  var pseudoRegisters = List.fill(taps.length)(0)

  def poke(value: Int): Int = {
    pseudoRegisters = value :: pseudoRegisters.take(taps.length - 1)
    var accumulator = 0
    for(i <- taps.indices) {
      accumulator += taps(i) * pseudoRegisters(i)
    }
    accumulator
  }
}

### Seq
请注意 `taps` 已经变成一个 `Seq[Int]`,这意味着类的用户在构造类时可以传递一个任意长度的 `Int` 序列。
### 寄存器
使用 `  var pseudoRegisters = List.fill(taps.length)(0)` 我们创建一个 `List` 来保存来自之前周期的值。选择 `List` 是因为将元素添加到头部并删除最后一个元素的语法非常简单。scala 集合家族的几乎任何成员都可以使用。我们还将这个列表初始化为包含全零。
### 注入
我们的类添加了一个 `poke` 函数/方法,它模拟将新输入放入滤波器并循环时钟。
### 更新寄存器
行 `pseudoRegisters = 值 :: pseudoRegisters.take(taps.length - 1)` 首先使用列表的 `take` 方法保留除最后一个元素外的所有元素,然后使用 `::` 列表连接运算符将 `值` 添加到缩减版本列表的头部。
### 计算输出
一个带有累加器的简单 for 循环将列表的每个元素与其对应的抽头系数相乘求和。只有 `accumulator` 的那行将该值作为函数结果返回。
## 调整之前的测试来测试我们的黄金模型
现在我们将使用之前的工作来确认我们的黄金模型有效。一点编辑魔法将我们之前的测试工具转换成了...

In [ ]:
val filter = new ScalaFirFilter(Seq(1, 1, 1, 1))

var out = 0

out = filter.poke(1)
println(s"out = $out")
assert(out == 1)  // 1, 0, 0, 0

out = filter.poke(4)
assert(out == 5)  // 4, 1, 0, 0
println(s"out = $out")

out = filter.poke(3)
assert(out == 8)  // 3, 4, 1, 0
println(s"out = $out")

out = filter.poke(2)
assert(out == 10)  // 2, 3, 4, 1
println(s"out = $out")

out = filter.poke(7)
assert(out == 16)  // 7, 2, 3, 4
println(s"out = $out")

out = filter.poke(0)
assert(out == 12)  // 0, 7, 2, 3
println(s"out = $out")

执行前面的块表明我们的软件模型返回与 My4ElementFir 相同的结果。


## 使用黄金模型测试电路<a name="use-golden-model-as-测试"></a> 
现在我们对我们的黄金模型有相当大的信心,我们重写我们的测试来比较电路输出与黄金模型的输出,而不是使用繁琐的手工示例。
下面是一个快速的第一遍方法。

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new My4ElementFir(1, 1, 1, 1)) { c =>
    for(i <- 0 until 100) {
        val input = scala.util.Random.nextInt(8)

        val goldenModelResult = goldenModel.poke(input)

        c.io.in.poke(input.U)

        c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

        c.clock.step(1)
    }

}


我们的测试运行 100 个周期,并检查两种不同的方法,硬件和软件,在每个步进时是否同步。

### 需要注意的事项
(即我们在编写此内容时实际犯的错误。)

1. 将步进放在正确的位置。软件和硬件的执行方式不同;很容易弄错。
1. 这个测试很弱,因为它对 IO 和寄存器的大小非常敏感。实现一个观察任意数据位宽包装行为的软件黄金模型可能很复杂。这里我们只确保我们只传入适合的值。

<span style="color:blue">**示例: 参数化 FIR 生成器**</span><br><a name="fir-golden-model"></a> 
下面我们创建了一个新的 Filter 类,`MyManyElementsFilter`,它接收一个常量 `Seq` 作为抽头。这个列表可以是任意数量的元素。
为了更完善,添加了一个 `bitWidth`,它允许我们控制电路可以处理的数字大小。
为了响应可变长度,我们必须重构寄存器的创建及其连接方式。
下面使用的方法使用了可用集合函数库的一个简单子集。
后面的部分展示了如何更简洁地表达行为,同时也使发生的事情更清晰。

In [ ]:
class MyManyElementFir(consts: Seq[Int], bitWidth: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt(bitWidth.W))
  })

  val regs = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      if(i == 0) regs += io.in
      else       regs += RegNext(regs(i - 1), 0.U)
  }
  
  val muls = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      muls += regs(i) * consts(i).U
  }

  val scan = mutable.ArrayBuffer[UInt]()
  for(i <- 0 until consts.length) {
      if(i == 0) scan += muls(i)
      else scan += muls(i) + scan(i - 1)
  }

  io.out := scan.last
}

#### 我们是如何做到的
有三个并行部分分别从第 7、13 和 18 行开始。
我们使用了一个名为 `ArrayBuffer` 的 Scala 集合类型。
`ArrayBuffer` 允许您使用 `+=` 运算符追加元素(也可以插入和删除,但我们不需要这个)。
首先,我们创建一个 ArrayBuffer `regs`,其元素将是 `UInt`。
然后迭代抽头,添加输入作为第一个元素,随后使用 RegNext 创建寄存器,它将寄存器的输入连接到前一个元素(`regs(i-1)`)并将其初始化为无符号零(`0.U`)。
这些寄存器将保存所需的输入先前值。

接下来,我们创建另一个 `UInt` 的 ArrayBuffer `muls`。
muls 的每个元素将是一个节点,其第 i 个元素是 `regs(i)` 和 `const(i)` 的乘积。

注意 `scan.last` 方法的使用。
它取集合的最后一个元素,是在 `regs` 构建期间使用的 `regs(i - 1)` 的更优雅的替代方案。

### 它的行为是否与 `My4ElementFir` 相同？
我们新版本的一个好的第一个测试是看它是否能通过我们刚刚应用于
`My4ElementFir` 的测试。
我们创建一个 `MyManyElementFir` 的实例,并运行更多数据通过它。

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new MyManyElementFir(Seq(1, 1, 1, 1), 8)) { c =>
    for(i <- 0 until 100) {
      val input = scala.util.Random.nextInt(8)

      val goldenModelResult = goldenModel.poke(input)

      c.io.in.poke(input.U)

      c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

      c.clock.step(1)
    }
}

### 现在让我们测试一堆不同大小的 FIR 滤波器
我们创建一些辅助函数:`r` 获取随机数;`runOneTest` 为一组特定的抽头创建黄金模型和滤波器的硬件仿真,然后运行至少两倍抽头数量的数据通过滤波器。

In [ ]:
/** 一个获取随机整数的便捷方法
  */
def r(): Int = {
  scala.util.Random.nextInt(1024)
}

/**
  * 运行比较软件和硬件滤波器的测试
  * 运行至少两倍于抽头数量的样本
  */
def runOneTest(taps: Seq[Int]) {
    val goldenModel = new ScalaFirFilter(taps)

    test(new MyManyElementFir(taps, 32)) { c =>
        for(i <- 0 until 2 * taps.length) {
            val input = r()

            val goldenModelResult = goldenModel.poke(input)

            c.io.in.poke(input.U)

            c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

            c.clock.step(1)
        }
    }
}

for(tapSize <- 2 until 100 by 10) {
    val taps = Seq.fill(tapSize)(r())  // 创建随机系数序列

    runOneTest(taps)
}

### 为了好玩,让我们做一个更大的
以下将在一个 500 抽头
FIR 滤波器上运行单个测试。可能需要一分钟或更长时间才能运行。
(提示:执行完成后,注意工具栏上的 Scala ● 变为 Scala ○。)

In [ ]:
runOneTest(Seq.fill(500)(r()))

---
# 硬件集合

<span style="color:blue">**示例:向我们的 FIR 添加运行时可配置的抽头**</span><br>
以下代码为我们的 FIR 生成器的 IO 添加了一个额外的 `consts` 向量,它允许在电路生成后从外部更改系数。
这是使用 Chisel 集合类型 `Vec` 完成的。
`Vec` 支持许多 scala 集合方法,但它只能包含 Chisel 硬件元素。
`Vec` 应该只在普通 Scala 集合不起作用的情况下使用。
基本上,这适用于以下两种情况之一。
1. 您需要一个束中的元素集合,通常是将用作 IO 的束。
1. 您需要通过索引访问集合,即它是硬件的一部分(想想寄存器文件)。


In [ ]:
class MyManyDynamicElementVecFir(length: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val out = Output(UInt(8.W))
    val consts = Input(Vec(length, UInt(8.W)))
  })

  // 参考解决方案
  val regs = RegInit(VecInit(Seq.fill(length - 1)(0.U(8.W))))
  for(i <- 0 until length - 1) {
      if(i == 0) regs(i) := io.in
      else       regs(i) := regs(i - 1)
  }
  
  val muls = Wire(Vec(length, UInt(8.W)))
  for(i <- 0 until length) {
      if(i == 0) muls(i) := io.in * io.consts(i)
      else       muls(i) := regs(i - 1) * io.consts(i)
  }

  val scan = Wire(Vec(length, UInt(8.W)))
  for(i <- 0 until length) {
      if(i == 0) scan(i) := muls(i)
      else scan(i) := muls(i) + scan(i - 1)
  }

  io.out := scan(length - 1)
}

In [ ]:
val goldenModel = new ScalaFirFilter(Seq(1, 1, 1, 1))

test(new MyManyDynamicElementVecFir(4)) { c =>
    c.io.consts(0).poke(1.U)
    c.io.consts(1).poke(1.U)
    c.io.consts(2).poke(1.U)
    c.io.consts(3).poke(1.U)
    for(i <- 0 until 100) {
        val input = scala.util.Random.nextInt(8)

        val goldenModelResult = goldenModel.poke(input)

        c.io.in.poke(input.U)

        c.io.out.expect(goldenModelResult.U, s"i $i, input $input, gm $goldenModelResult, ${c.io.out.peek().litValue}")

        c.clock.step(1)
    }
}


<span style="color:red">**练习: 32 位 RISC-V 处理器**</span><br><a name="fir-golden-model"></a>

[寄存器文件](https://en.wikipedia.org/wiki/Register_file)是制作处理器的重要构建块。
寄存器文件是一个寄存器数组,可以通过多个读或写端口从其读取或向其写入。
每个端口由地址和数据字段组成。

[RISC-V 指令集架构](https://riscv.org/specifications/)定义了几种变体,其中最简单的称为 RV32I。
RV32I 有一个大小为 32 的 32 位寄存器数组。
**索引 0 处的寄存器(第一个寄存器)在读取时始终为零,无论您向其写入什么**(拥有 0 很有用)。

为 RV32I 实现一个具有单个写端口和可参数化数量的读端口的寄存器文件。
只有在断言 `wen`(写使能)时才会执行写操作。

In [ ]:
class RegisterFile(readPorts: Int) extends Module {
    require(readPorts >= 0)
    val io = IO(new Bundle {
        val wen   = Input(Bool())
        val waddr = Input(UInt(5.W))
        val wdata = Input(UInt(32.W))
        val raddr = Input(Vec(readPorts, UInt(5.W)))
        val rdata = Output(Vec(readPorts, UInt(32.W)))
    })
    
    // UInt 向量的寄存器
    val reg = RegInit(VecInit(Seq.fill(32)(0.U(32.W))))
    
    ???

    
}

In [ ]:
test(new RegisterFile(2) ) { c =>
  def readExpect(addr: Int, value: Int, port: Int = 0): Unit = {
    c.io.raddr(port).poke(addr.U)
    c.io.rdata(port).expect(value.U)
  }
  def write(addr: Int, value: Int): Unit = {
    c.io.wen.poke(true.B)
    c.io.wdata.poke(value.U)
    c.io.waddr.poke(addr.U)
    c.clock.step(1)
    c.io.wen.poke(false.B)
  }
  // 初始化时所有值都应为 0
  for (i <- 0 until 32) {
    readExpect(i, 0, port = 0)
    readExpect(i, 0, port = 1)
  }

  // 写入 5 * 地址 + 3
  for (i <- 0 until 32) {
    write(i, 5 * i + 3)
  }

  // 检查写入是否成功
  for (i <- 0 until 32) {
    readExpect(i, if (i == 0) 0 else 5 * i + 3, port = i % 2)
  }
}

<div id="container"><section id="accordion"><div>
<输入 类型="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
    when (io.wen) {
        reg(io.waddr) := io.wdata
    }
    for (i &lt;- 0 until readPorts) {
        when (io.raddr(i) === 0.U) {
            io.rdata(i) := 0.U
        } .otherwise {
            io.rdata(i) := reg(io.raddr(i))
        }
    }

</pre></article></div></section></div>

---
# 完成了！

[返回顶部](#top)